# Method 2: StyleGAN2 inversion (unsupervised)
Pretrained StyleGAN2-ADA AFHQ generator. For each image optimise the latent w on the visible pixels only (MSE + LPIPS), then composite the hole. Slow (an optimisation loop per image); evaluate on a subset. Also the base for the optional pet-code personalisation.

**Code lives in** `src/methods/stylegan_inversion.py`. This notebook only drives it: train / run, evaluate, save results.

Status: method file is a stub; the cells below run once it is implemented.

## 0. Setup

In [ ]:
import os, sys, subprocess
if 'google.colab' in sys.modules and not os.path.exists('DLpastiA'):
    subprocess.run(['git','clone','-b','claude/upbeat-rubin-7at9xc','https://github.com/EricaCleo/DLpastiA.git','DLpastiA'])
    os.chdir('DLpastiA')
elif os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
%pip install -q kagglehub lpips scikit-image torch-fidelity pyyaml tqdm pandas matplotlib
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print(os.getcwd(), '| torch', torch.__version__, '| device:', dev)
if dev == 'cpu': print('NO GPU: on Colab use Runtime > Change runtime type > GPU')

## 1. Data

In [ ]:
# Data (run notebook 00 once first, or this downloads it). Needs env var KAGGLE_API_TOKEN.
from pathlib import Path
if not Path('data/raw').exists():
    !{sys.executable} scripts/download_data.py
from src.data import get_datasets, get_loaders, load_config
cfg = load_config()
train_loader, _ = get_loaders(batch_size=16)
train_ds, test_ds = get_datasets(cfg)
print(len(train_ds), len(test_ds))

## 2. Load generator

In [ ]:
from src.methods.stylegan_inversion import StyleGANInversion
inv = StyleGANInversion(generator_pkl='checkpoints/stylegan2-ada-afhq.pkl', steps=800, device=dev)

## 3. Evaluate (K=5 fills, subset because it is slow)

In [ ]:
from src.eval import evaluate, save_result
for mask in ['square', 'one_sided', 'centre']:
    res = evaluate(inv, mask, n_images=100, device=dev, save_dir=f'outputs/samples/{inv.name}_{mask}')
    save_result(inv.name, mask, res)
    print(mask, res)

## 4. (Optional) personalisation with a pet code

In [ ]:
# w_pet = mean of inverted latents of the pet's photos
# inv.invert(masked, mask, w_init=w_pet)